# nb03_cpx_baseline_models (v10)

Thin orchestrator around scripts/v10_phase_d_cpx_runner.py.

**Prerequisite:** results/v10_optuna_best_params_cpx.json exists. This is produced by scripts/v10_phase_d_cpx_driver.py (run in Phase D).

**What this notebook does:**
1. Verify best-params manifest is present.
2. Run the cpx runner: refit 8 bases, build OOF, fit 3 internal ensembles, run T01/T02/T11/T12 per (target, track, feature_set) cell.
3. Display per-cell base and ensemble test-set RMSE/R^2 tables.
4. Display T01-T12 test summary.

**Pre-registered tests (computed here):**
- T01: Boosted is primary
- T02: Ensemble beats best base (4 methods compared)
- T11: CatBoost or LightGBM beats XGB (NEW in v10)
- T12: MLP or ElasticNet beats tree models (NEW in v10)

T03-T10 require v9 mechanics (resampling, bias correction, OOD) and run in a follow-up notebook.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from config import RESULTS
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)


In [ ]:
best_json = RESULTS / 'v10_optuna_best_params_cpx.json'
assert best_json.exists(), f'{best_json} missing. Run scripts/v10_phase_d_cpx_driver.py first.'

import json
with open(best_json) as f:
    payload = json.load(f)
print(f'Phase D manifest: n_results={len(payload["results"])}  n_failures={len(payload.get("failures", []))}')
print(f'Generated: {payload["generated_at"]}  n_trials={payload["n_trials"]}')


In [ ]:
# Run the cpx runner (refit + ensembles + T01/T02/T11/T12).
# Duration: ~15-30 min across 12 cells.
import subprocess
r = subprocess.run(['python', 'scripts/v10_phase_d_cpx_runner.py'],
                   capture_output=True, text=True, cwd=str(ROOT))
print('
'.join(r.stdout.splitlines()[-30:]))
if r.returncode != 0:
    print('STDERR:'); print(r.stderr[-2000:])
assert r.returncode == 0


In [ ]:
df_bases = pd.read_csv(RESULTS / 'v10_cpx_per_cell_results.csv')
print(f'n_rows={len(df_bases)}')
df_bases.round(3)


In [ ]:
df_ens = pd.read_csv(RESULTS / 'v10_cpx_ensemble_results.csv')
df_ens.round(3)


In [ ]:
idx = df_bases.groupby(['target', 'track'])['test_rmse'].idxmin()
best_per_cell = df_bases.loc[idx].reset_index(drop=True)
print('Best base model per (target, track):')
best_per_cell[['target', 'track', 'model', 'feature_set', 'test_rmse', 'test_r2']].round(3)


In [ ]:
idx = df_ens.groupby(['target', 'track'])['test_rmse'].idxmin()
best_ens = df_ens.loc[idx].reset_index(drop=True)
print('Best ensemble per (target, track):')
best_ens.round(3)


In [ ]:
from scripts.v10_test_protocol import summarize_tests
tests_df = summarize_tests(pipeline='cpx')
print(f'n_test_rows={len(tests_df)}')
tests_df[['test_id', 'target', 'track', 'passed', 'value', 'threshold']].head(60)


In [ ]:
if not tests_df.empty:
    rate = tests_df.groupby('test_id')['passed'].agg(['sum', 'count'])
    rate['pass_rate'] = rate['sum'] / rate['count']
    print('Pass rate per test (across cells):')
    print(rate)


## Next steps

- If T01-T12 subset passes: commit canonical models under models/canonical/cpx/ and proceed to Phase E (twopx rebuild).
- If any test fails: investigate per-cell details in results/v10_cpx_per_cell_results.csv and the full log at results/v10_cpx_test_log.csv.
- T03-T10 follow-up tests require v9 mechanics (resampling, bias correction, OOD) and run in a separate notebook once refit-level results are stable.